# Tahap 4 - Case Solution Reuse

Project: Case-Based Reasoning untuk Pidana Umum - Pencurian di PN Tangerang

Notebook ini digunakan sebagai bagian dari pipeline CBR.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import json
import joblib

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

BASE_DIR = Path("..").resolve()

PROCESSED_DIR = BASE_DIR / "data" / "processed"
EVAL_DIR = BASE_DIR / "data" / "eval"
RESULTS_DIR = BASE_DIR / "data" / "results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

cases_path = PROCESSED_DIR / "cases.csv"
vectorizer_path = PROCESSED_DIR / "tfidf_vectorizer.joblib"
matrix_path = PROCESSED_DIR / "tfidf_matrix.joblib"

cases_df = pd.read_csv(cases_path, dtype=str).fillna("")

print("Jumlah kasus:", len(cases_df))

cases_df[[
    "case_id",
    "no_perkara",
    "terdakwa",
    "amar_lainnya",
    "solution_label",
    "lama_pidana"
]].head()

Jumlah kasus: 40


,case_id,no_perkara,terdakwa,amar_lainnya,solution_label,lama_pidana
0,case_001,1022/Pid.B/2010/PN.TNG,JAMALUDIN Bin SANIF terbukti secara sah danmey...,,Lain-lain,
1,case_002,497 / PID.B / 2014 / PN.TNG.,AMIN Als. UBE Bin UDIN danALIP KURNIAWAN Als. ...,,Lain-lain,
2,case_003,1885 /Pid.B/2011/PN.TNG,MAULANAHASANUDIN als. KEDOK binROJALI telah te...,,Lain-lain,
3,case_004,1073/Pid.B/2019/PN Tng,MUHAMMAD RIKI YAKUB Alias INYONG Bin RIPIN 37 ...,,Lain-lain,
4,case_005,678/ PID.B/ 2011/ PN TNG,di persidangan ;Telah mendengar pembacaan tunt...,,Lain-lain,


In [2]:
def clean_text_for_retrieval(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\s./-]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def build_retrieval_text(row):
    parts = [
        row.get("no_perkara", ""),
        row.get("pengadilan", ""),
        row.get("jenis_perkara", ""),
        row.get("terdakwa", ""),
        row.get("pasal", ""),
        row.get("amar_lainnya", ""),
        row.get("catatan_amar", ""),
        row.get("ringkasan_fakta", ""),
        row.get("argumen_hukum", ""),
        row.get("solution_text", ""),
        row.get("text_full", "")
    ]
    
    return clean_text_for_retrieval(" ".join([str(p) for p in parts if str(p).strip() != ""]))


if "retrieval_text" not in cases_df.columns:
    cases_df["retrieval_text"] = cases_df.apply(build_retrieval_text, axis=1)

cases_df[["case_id", "retrieval_text"]].head()

,case_id,retrieval_text
0,case_001,1022/pid.b/2010/pn.tng pn tangerang pidana umu...
1,case_002,497 / pid.b / 2014 / pn.tng. pn tangerang pida...
2,case_003,1885 /pid.b/2011/pn.tng pn tangerang pidana um...
3,case_004,1073/pid.b/2019/pn tng pn tangerang pidana umu...
4,case_005,678/ pid.b/ 2011/ pn tng pn tangerang pidana u...


In [3]:
if vectorizer_path.exists() and matrix_path.exists():
    vectorizer = joblib.load(vectorizer_path)
    tfidf_matrix = joblib.load(matrix_path)
    print("Vectorizer dan matrix berhasil diload.")
else:
    print("Vectorizer belum ada. Membuat ulang TF-IDF...")
    
    vectorizer = TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        max_features=5000,
        min_df=1
    )
    
    tfidf_matrix = vectorizer.fit_transform(cases_df["retrieval_text"])
    
    joblib.dump(vectorizer, vectorizer_path)
    joblib.dump(tfidf_matrix, matrix_path)
    
    print("Vectorizer dan matrix berhasil dibuat ulang.")

print("Ukuran matrix:", tfidf_matrix.shape)

Vectorizer dan matrix berhasil diload.
Ukuran matrix: (40, 5000)


In [4]:
def retrieve(query: str, k: int = 5):
    query_clean = clean_text_for_retrieval(query)
    
    query_vector = vectorizer.transform([query_clean])
    
    similarities = cosine_similarity(query_vector, tfidf_matrix).flatten()
    
    top_indices = similarities.argsort()[::-1][:k]
    
    results = cases_df.iloc[top_indices].copy()
    results["similarity_score"] = similarities[top_indices]
    
    return results

In [5]:
case_solutions = {}

for _, row in cases_df.iterrows():
    case_id = row["case_id"]
    
    solution_text = row.get("solution_text", "")
    
    if solution_text == "":
        solution_text = row.get("catatan_amar", "")
    
    if solution_text == "":
        solution_text = row.get("amar_lainnya", "")
    
    if solution_text == "":
        solution_text = row.get("text_full", "")
    
    case_solutions[case_id] = {
        "solution_label": row.get("solution_label", "Lain-lain"),
        "solution_text": solution_text,
        "amar_lainnya": row.get("amar_lainnya", ""),
        "lama_pidana": row.get("lama_pidana", ""),
        "no_perkara": row.get("no_perkara", "")
    }

case_solutions_path = PROCESSED_DIR / "case_solutions.json"

with open(case_solutions_path, "w", encoding="utf-8") as f:
    json.dump(case_solutions, f, ensure_ascii=False, indent=2)

print("case_solutions.json berhasil dibuat:")
print(case_solutions_path)

case_solutions.json berhasil dibuat:
/home/zack/Penalaran-Komputer-subcpmk-3/data/processed/case_solutions.json


In [6]:
def weighted_vote_solution(top_k_df):
    label_scores = {}
    
    for _, row in top_k_df.iterrows():
        label = row.get("solution_label", "Lain-lain")
        score = float(row.get("similarity_score", 0))
        
        if label not in label_scores:
            label_scores[label] = 0
        
        label_scores[label] += score
    
    if not label_scores:
        return "Lain-lain"
    
    predicted_label = max(label_scores, key=label_scores.get)
    
    return predicted_label


def predict_outcome(query: str, k: int = 5):
    top_k = retrieve(query, k=k)
    
    predicted_label = weighted_vote_solution(top_k)
    
    # Ambil solusi teks dari kasus paling mirip
    best_case = top_k.iloc[0]
    best_case_id = best_case["case_id"]
    
    best_solution = case_solutions.get(best_case_id, {})
    
    predicted_solution_text = best_solution.get("solution_text", "")
    predicted_lama_pidana = best_solution.get("lama_pidana", "")
    
    top_case_ids = top_k["case_id"].tolist()
    top_scores = top_k["similarity_score"].round(4).tolist()
    
    return {
        "predicted_solution_label": predicted_label,
        "predicted_solution_text": predicted_solution_text,
        "predicted_lama_pidana": predicted_lama_pidana,
        "best_case_id": best_case_id,
        "top_5_case_ids": top_case_ids,
        "top_5_scores": top_scores,
        "top_k_detail": top_k[[
            "case_id",
            "no_perkara",
            "terdakwa",
            "solution_label",
            "lama_pidana",
            "similarity_score"
        ]]
    }

In [7]:
query = """
Terdakwa melakukan tindak pidana pencurian dalam keadaan memberatkan.
Barang bukti dikembalikan kepada korban dan terdakwa dijatuhi pidana penjara.
"""

prediction = predict_outcome(query, k=5)

print("Predicted Label:", prediction["predicted_solution_label"])
print("Predicted Lama Pidana:", prediction["predicted_lama_pidana"])
print("Best Case:", prediction["best_case_id"])
print("Top 5 Case IDs:", prediction["top_5_case_ids"])
print("Top 5 Scores:", prediction["top_5_scores"])

prediction["top_k_detail"]

Predicted Label: Lain-lain
Predicted Lama Pidana: 
Best Case: case_006
Top 5 Case IDs: ['case_006', 'case_003', 'case_015', 'case_032', 'case_031']
Top 5 Scores: [0.1577, 0.1476, 0.1323, 0.1292, 0.1212]


,case_id,no_perkara,terdakwa,solution_label,lama_pidana,similarity_score
5,case_006,1527/Pid.B/2014/PN.TNG,"NUR APRIYANI Binti (Alm) NURDIN, terbuktibersa...",Lain-lain,,0.157701
2,case_003,1885 /Pid.B/2011/PN.TNG,MAULANAHASANUDIN als. KEDOK binROJALI telah te...,Lain-lain,,0.147631
14,case_015,1427/Pid.B/2022/PN Tng,HASAN FUAD Bin DAYAT 41 — 1 MENGADILI: Menyata...,Lain-lain,,0.132340
31,case_032,168/Pid.B/2023/PN Tng,PANDU ASMIARDI 57 — 4 Menyatakan Terdakwa PAND...,Lain-lain,,0.129225
30,case_031,1686/Pid.B/2022/PN Tng,1.FAHRU ROJI RIZAL FAUZI als OJI Bin alm MATSO...,Lain-lain,,0.121178


In [8]:
queries_path = EVAL_DIR / "queries.json"

if queries_path.exists():
    with open(queries_path, "r", encoding="utf-8") as f:
        eval_queries = json.load(f)
else:
    eval_queries = [
        {
            "query_id": "manual_001",
            "query": "Terdakwa melakukan pencurian dan dijatuhi pidana penjara.",
            "ground_truth_case_id": "",
            "ground_truth_solution_label": ""
        },
        {
            "query_id": "manual_002",
            "query": "Kasus pencurian dengan barang bukti dikembalikan kepada korban.",
            "ground_truth_case_id": "",
            "ground_truth_solution_label": ""
        }
    ]

prediction_rows = []

for item in eval_queries:
    query_id = item.get("query_id", "")
    query_text = item.get("query", "")
    
    pred = predict_outcome(query_text, k=5)
    
    prediction_rows.append({
        "query_id": query_id,
        "query": query_text,
        "ground_truth_case_id": item.get("ground_truth_case_id", ""),
        "ground_truth_solution_label": item.get("ground_truth_solution_label", ""),
        "predicted_solution_label": pred["predicted_solution_label"],
        "predicted_solution_text": pred["predicted_solution_text"],
        "predicted_lama_pidana": pred["predicted_lama_pidana"],
        "best_case_id": pred["best_case_id"],
        "top_5_case_ids": ", ".join(pred["top_5_case_ids"]),
        "top_5_scores": ", ".join([str(s) for s in pred["top_5_scores"]])
    })

predictions_df = pd.DataFrame(prediction_rows)

predictions_path = RESULTS_DIR / "predictions.csv"
predictions_df.to_csv(predictions_path, index=False)

print("Prediksi berhasil disimpan:")
print(predictions_path)

predictions_df

Prediksi berhasil disimpan:
/home/zack/Penalaran-Komputer-subcpmk-3/data/results/predictions.csv


,query_id,query,ground_truth_case_id,ground_truth_solution_label,predicted_solution_label,predicted_solution_text,predicted_lama_pidana,best_case_id,top_5_case_ids,top_5_scores
0,eval_001,Pengadilan PN TANGERANG Pidana Umum Pencurian ...,case_020,Lain-lain,Lain-lain,Pengadilan PN TANGERANG Pidana Umum Pencurian ...,,case_020,"case_020, case_018, case_033, case_037, case_038","0.8744, 0.3464, 0.2744, 0.2585, 0.2533"
1,eval_002,Pengadilan PN TANGERANG Pidana Umum Register :...,case_017,Lain-lain,Lain-lain,Pengadilan PN TANGERANG Pidana Umum Register :...,,case_017,"case_017, case_010, case_016, case_033, case_002","0.9297, 0.1242, 0.1215, 0.1204, 0.1171"
2,eval_003,Pengadilan PN TANGERANG Pidana Umum Register :...,case_016,Lain-lain,Lain-lain,Pengadilan PN TANGERANG Pidana Umum Register :...,,case_016,"case_016, case_007, case_015, case_008, case_032","0.9202, 0.1379, 0.1324, 0.13, 0.1281"
3,eval_004,Pengadilan PN TANGERANG Pidana Umum Register :...,case_027,Lain-lain,Lain-lain,Pengadilan PN TANGERANG Pidana Umum Register :...,,case_027,"case_027, case_031, case_015, case_039, case_026","0.9831, 0.3037, 0.253, 0.2258, 0.2195"
4,eval_005,Pengadilan PN TANGERANG Pidana Umum Pencurian ...,case_005,Lain-lain,Lain-lain,Pengadilan PN TANGERANG Pidana Umum Pencurian ...,,case_005,"case_005, case_003, case_033, case_037, case_006","0.8864, 0.1861, 0.1572, 0.1555, 0.146"
5,eval_006,Pengadilan PN TANGERANG Pidana Umum Register :...,case_013,Lain-lain,Lain-lain,Pengadilan PN TANGERANG Pidana Umum Register :...,,case_013,"case_013, case_011, case_023, case_015, case_012","0.9822, 0.146, 0.1456, 0.1438, 0.1399"
6,eval_007,Pengadilan PN TANGERANG Pidana Umum Pencurian ...,case_038,Lain-lain,Lain-lain,Pengadilan PN TANGERANG Pidana Umum Pencurian ...,,case_038,"case_038, case_020, case_018, case_002, case_003","0.9542, 0.3592, 0.283, 0.2043, 0.1336"
7,eval_008,Pengadilan PN TANGERANG Pidana Umum Pencurian ...,case_028,Lain-lain,Lain-lain,Pengadilan PN TANGERANG Pidana Umum Pencurian ...,,case_028,"case_028, case_023, case_015, case_026, case_032","0.982, 0.2588, 0.2576, 0.246, 0.2249"
8,eval_009,Pengadilan PN TANGERANG Pidana Umum Kejahatan ...,case_040,Lain-lain,Lain-lain,Pengadilan PN TANGERANG Pidana Umum Kejahatan ...,,case_040,"case_040, case_001, case_003, case_025, case_036","0.99, 0.1791, 0.1224, 0.1138, 0.0923"
9,eval_010,Pengadilan PN TANGERANG Pidana Umum Pencurian ...,case_007,Lain-lain,Lain-lain,Pengadilan PN TANGERANG Pidana Umum Pencurian ...,,case_007,"case_007, case_016, case_008, case_037, case_030","0.9521, 0.1623, 0.1401, 0.1386, 0.1188"


In [9]:
print("File output tahap 4:")

for path in [
    case_solutions_path,
    predictions_path
]:
    print(path, "=>", path.exists())

File output tahap 4:
/home/zack/Penalaran-Komputer-subcpmk-3/data/processed/case_solutions.json => True
/home/zack/Penalaran-Komputer-subcpmk-3/data/results/predictions.csv => True
